In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.vector_ar.var_model import VAR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, root_mean_squared_error

# 1. Veri Setini Yükleme
file_path = "data/selected_lip_coordinates.csv"
df = pd.read_csv(file_path)

# 2. Eksik Veri Kontrolü
print("Missing values:")
print(df.isnull().sum())

# Zaman bilgisini x ekseni olarak belirle
#time = df.index

# 3. Zaman Serisi Görselleştirme
#plt.figure(figsize=(12, 6))

# Her sütunu ayrı ayrı çiz
#for col in df.columns:
    #plt.plot(time, df[col], label=col)

# Grafik ayarları
#plt.xlabel("Time")
#plt.ylabel("Value")
#plt.title("Time Series Plot of Multiple Features")
#plt.legend(title="Features")
#plt.grid(True)
#plt.show()
#plt.close()

Missing values:
time    0
0_x     0
13_x    0
14_x    0
17_x    0
37_x    0
0_y     0
13_y    0
14_y    0
17_y    0
37_y    0
dtype: int64


In [2]:
# 4. ADF Durağanlık Testi ve Durağan Olmayanlara Fark Alma
#def adf_test(series):
    #result = adfuller(series.dropna())
    #print(f"ADF Statistic: {result[0]}")
    #print(f"p-value: {result[1]}")
    #print("Critical Values:")
    #for key, value in result[4].items():
        #print(f"   {key}: {value}")
    #if result[1] <= 0.05:
        #print("Seri durağandır.")
        #return True
    #else:
        #print("Seri durağan değildir.")
        #return False

#print("Durağanlık Testi Sonuçları:")
#non_stationary_columns = []
#for column in df.columns:
    #print(f"\n{column} için ADF Testi:")
    #is_stationary = adf_test(df[column])
    #if not is_stationary:
        #non_stationary_columns.append(column)

#print(f"Durağan olmayan sütunlar: {non_stationary_columns}")

In [3]:
# 5. ACF ve PACF Grafikleri
##fig, ax = plt.subplots(2, 1, figsize=(12, 8))
#plot_acf(df.dropna().iloc[:, 0], ax=ax[0], lags=20)
#ax[0].set_title("Autocorrelation Function (ACF)")
#plot_pacf(df.dropna().iloc[:, 0], ax=ax[1], lags=20)
#ax[1].set_title("Partial Autocorrelation Function (PACF)")
#plt.subplots_adjust(hspace=0.7, wspace=0.4)
#plt.show()

In [11]:
# Veri Setini Train ve Test olarak Bölme
# 'time' sütununu çıkar ve yalnızca koordinatları modele dahil et
data = df.drop(columns=['time'])
train_size = int(len(data) * 0.8)  # %80 eğitim, %20 test
train, test = data[0:train_size], data[train_size:len(data)]

# Veriyi float türüne dönüştür (önemli!)
train = train.astype('float64')
test = test.astype('float64')

# VARMAX modelini oluştur ve eğit
model = VARMAX(train, exog=None, order=(1, 1))
model_fit = model.fit(disp=False)

# Test seti üzerinde tahmin yapma
forecast = model_fit.forecast(steps=len(test))

# Tahminleri DataFrame'e dönüştürme
forecast_df = pd.DataFrame(forecast.values, index=test.index, columns=test.columns)


# Performans Metriklerini Hesaplama

mae = mean_absolute_error(test, forecast_df)
mse = mean_squared_error(test, forecast_df)
rmse = root_mean_squared_error(test, forecast_df)
mape = mean_absolute_percentage_error(test, forecast_df)

print("Gerçek Değerler vs Tahmin Edilen Değerler:")
for expected, predicted in zip(test.values, forecast_df.values):
    print(f"Gerçek: {expected}, \n\nTahmin: {predicted}")
    print("------------------------------------------------------------------------------")

print("------------------------------------------------------------------------------")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
# plot forecasts against actual outcomes
# plt.plot(test)
# plt.plot(predictions, color='red')
# plt.show()

C:\Users\Gozde\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\Gozde\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Gerçek Değerler vs Tahmin Edilen Değerler:
Gerçek: [631. 632. 632. 633. 621. 326. 337. 337. 352. 325.], 

Tahmin: [631.16846885 631.71389067 632.24576848 633.05230791 621.64814813
 325.5766404  335.11573947 336.45770437 350.55076837 324.30865937]
------------------------------------------------------------------------------
Gerçek: [630. 630. 631. 632. 620. 326. 337. 337. 351. 325.], 

Tahmin: [631.13857833 631.74280125 632.20172836 632.91563734 621.72115999
 325.3332443  334.75674156 336.29036517 350.30882744 324.07277507]
------------------------------------------------------------------------------
Gerçek: [628. 629. 629. 630. 618. 325. 335. 337. 351. 324.], 

Tahmin: [631.15383445 631.74010707 632.18270966 632.85329116 621.78737488
 325.09625695 334.44133137 336.13697829 350.10482936 323.83250278]
------------------------------------------------------------------------------
Gerçek: [627. 628. 628. 629. 618. 323. 333. 337. 352. 322.], 

Tahmin: [631.17247952 631.74193051 632.174451

In [9]:
# Veri Setini Train ve Test olarak Bölme
# 'time' sütununu çıkar ve yalnızca koordinatları modele dahil et
#data = df.drop(columns=['time'])
#train_size = int(len(data) * 0.8)  # %80 eğitim, %20 test
#train, test = data[0:train_size], data[train_size:len(data)]

# Veriyi float türüne dönüştür (önemli!)
#train = train.astype('float64')
#test = test.astype('float64')

#history = train.values.tolist()
#predictions = []

# Walk-forward validation ile tahmin etme
#for t in range(len(test)):
    #model = VARMAX(pd.DataFrame(history, columns=data.columns), exog=None, order=(1, 1))
    #model_fit = model.fit(disp=False)
    #forecast = model_fit.forecast(steps=1)
    #predictions.append(forecast.values[0])
    #history.append(test.values[t])

# Tahminleri DataFrame'e dönüştürme
#predictions_df = pd.DataFrame(predictions, index=test.index, columns=test.columns)


# Performans Metriklerini Hesaplama

#mae = mean_absolute_error(test, predictions)
#mse = mean_squared_error(test, predictions)
#rmse = root_mean_squared_error(test, predictions)
#mape = mean_absolute_percentage_error(test, predictions)

#print("Gerçek Değerler vs Tahmin Edilen Değerler:")
#for expected, predicted in zip(test.values, predictions):
    #print(f"Gerçek: {expected}, \n\nTahmin: {predicted}")
    #print("------------------------------------------------------------------------------")

#print("------------------------------------------------------------------------------")
#print(f"Mean Absolute Error (MAE): {mae}")
#print(f"Mean Squared Error (MSE): {mse}")
#print(f"Root Mean Squared Error (RMSE): {rmse}")
#print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
# plot forecasts against actual outcomes
# # plt.plot(test)
# plt.plot(predictions, color='red')
# plt.show()